In [1]:
from langchain_dartmouth.llms import ChatDartmouthCloud
import pandas as pd
from glob import glob
import json
import numpy as np

In [2]:
# ChatDartmouthCloud.list(dartmouth_chat_api_key=key,)

In [3]:
import os
from getpass import getpass

# Set DARTMOUTH_CHAT_API_KEY in your environment (see .env.example), or enter it when prompted
key = os.environ.get("DARTMOUTH_CHAT_API_KEY") or getpass("Dartmouth Chat API key: ")
llm = ChatDartmouthCloud(model_name="openai.gpt-4.1-mini-2025-04-14", dartmouth_chat_api_key=key, temperature=0.3)

In [4]:
#Concatenate headlines to strings 
df2 = pd.concat([pd.read_csv(f) for f in 
                glob("../data/*.csv") ])
df2 = df2[~df2.title.isna()]

headlines_list = df2['title'].astype(str).tolist()

In [5]:
df3 = df2[df2.title.str.lower().str.contains("taiwan")]
df3 = df3.sample(frac=1.0,random_state=42) 

## Exploration (Topic Modeling using Large Language Model)

In [6]:
big_string =  "\n".join(df3.sample(frac=0.35).title.values)

prompt = """
Please identify the top 10 topics in all of these headlines. Each line break represents one new article:
"""

prompt += big_string

In [1]:
result = llm.invoke(prompt)
print(result.content)

## Classification (Topic Labeling)

In [26]:

label_system_prompt = '''
As a political researcher analyzing the portrayals of Taiwan in US media, your goal is to evaluate the topic and sentiment expressed in a provided headline. 
Each indicator should represent whether the post mentions, supports or opposes various themes associated with Taiwan.
Here is the headline:
'''

json_out = '''
You will need to output a JSON object. Return the structured JSON only, with no additional text, descriptions, or explanations.
{
    crisis: int, // Set to 1 if the headline suggests there is a crisis in Taiwan, otherwise 0,
    independence: int, // Set to 1 if the headline mentions Taiwanese independence, otherwise 0,
    unification: int, // Set to 1 if the headline mentions Chinese unification of Taiwan, otherwise 0,
    democratic_values int, // Set to 1 if the headline mentions democratic values shared with Taiwan, otherwise 0,
    natural_disaster: int, // Set to 1 if the headline is about some sort of natural disaster, otherwise 0,
    semiconductors: int, // Set to 1 if the headline mentions semiconductors, chips or related technology, otherwise 0,
    cultural: int, // Set to 1 if the headline mentions social movements, LGBTQ+ rights, or cultural events , otherwise 0,
    pro_Taiwan: int, // Set to 1 if the post supports Taiwan, otherwise 0,
    against_Taiwan: int // Set to 1 if the post opposes Taiwan, otherwise 0,
    pro_China: int, // Set to 1 if the post supports mainland China, otherwise 0,
    against_China: int, // Set to 1 if the post opposes mainland China, otherwise 0
}
'''

format_prompt = lambda t: label_system_prompt + "\n" + t + "\n" + json_out # returns the output

In [27]:
df3["prompt"] = df3.title.apply(format_prompt) # organizes all the prompt INPUT into one new column in df3
#df3.prompt

In [19]:
import os
os.makedirs("raw_labels", exist_ok=True)

# E = headlines already labeled (empty on a fresh run, so this no longer crashes)
_files = glob("raw_labels/*.pkl")
E = pd.concat([pd.read_pickle(f) for f in _files]) if _files else pd.DataFrame({"id": []})


In [20]:
df3 = df3[~df3.id.isin(E.id)] #removes from df3 any rows whose id appears in E. 

# df3 contains the headlines yet to go through prompt; E contains all headlines already gone through prompt.

In [ ]:
nchunks = 40
# split df3 into 40 sub-DataFrames (same split as before; avoids the deprecated/removed DataFrame.swapaxes path)
chunks = [df3.iloc[idx].copy() for idx in np.array_split(np.arange(len(df3)), nchunks)]


In [11]:
chunks[0].shape

(326, 9)

In [12]:
batch_name = "b4" # b2 onward is openai.gpt-4.1-mini-2025-04-14

for i,sub_df in enumerate(chunks):
    bb = llm.batch(list( sub_df.prompt.values ))#Extracts the list of prompts from sub_df, 
                                                #Sends the list to the LLM in batch mode & Stores the list of responses in bb.
    sub_df["llm_results"] = bb
    sub_df.to_pickle("raw_labels/{}-{}.pkl".format(batch_name,i))

In [24]:
# sub_df.to_pickle()

## Read and format

In [29]:
LABEL_KEYS = ["crisis", "independence", "unification", "democratic_values", "natural_disaster",
              "semiconductors", "cultural", "pro_Taiwan", "against_Taiwan", "pro_China", "against_China"]

def parse_response(r):
    """Parse the model reply as JSON; unparseable replies become NaN (and are dropped downstream, see 06_*)."""
    try:
        return json.loads(r.content)
    except Exception:
        return {k: np.nan for k in LABEL_KEYS}


In [38]:
labeled_df = pd.DataFrame( sub_df.llm_results.apply(parse_response).to_list() ) 
#parsing the LLM outputs into a clean structured format and turning that into a df

,crisis,independence,unification,democratic_values,natural_disaster,semiconductors,cultural,pro_Taiwan,against_Taiwan,pro_China,against_China
0,1,0,0,0,0,0,0,1,0,0,1
1,1,0,0,0,0,0,0,1,0,0,0
2,1,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,1,0,0,0
4,0,0,1,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...
515,0,0,0,0,0,1,0,1,0,0,0
516,1,0,0,0,0,0,0,1,0,0,1
517,0,0,0,0,0,0,0,1,0,0,1
518,0,0,0,0,0,0,0,1,0,0,0


In [ ]:
# [disabled: iterated DataFrame columns and used undefined `label`]
# headlines=[]
# 
# for h in df3:
#     headline_data= json.loads(label.content[8:-4])
#     headlines.append(headline_data)

In [170]:
response_list = [parse_response(r) for r in bb]

#creating a list of formatted response (content) from bb, which is the list of raw label results 


In [171]:
pd.DataFrame( response_list )

,crisis,independence,unification,democratic_values,natural_disaster,semiconductors,cultural,pro_Taiwan,against_Taiwan,pro_China,against_China
0,1,0,0,1,0,0,0,1,0,0,1
1,0,0,0,0,0,1,0,1,0,0,0
2,0,0,0,0,0,1,0,0,1,0,0
3,0,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,1,0,1,0,0,0


### Post process

In [ ]:
# [disabled: uses undefined `label`]
# b = json.loads(label.content[8:-4])
# b

In [ ]:
# [disabled: `label_text_json` / `bigass_string` are undefined]
# label = label_text_json(bigass_string)
# label.content